In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

### Scenario:
You're given `employees_hierarchy.json` — a flat employee list where each row references its manager via `manager_id` (a self-referencing structure, i.e. an **org chart**). Spark has no native recursive query support the way SQL's `WITH RECURSIVE` does, so flattening a hierarchy of unknown depth requires a different technique: **iteratively self-joining** the data, one level at a time, until you've walked all the way up to the root for every employee. This is a genuinely common real-world problem — org charts, bill-of-materials trees, category hierarchies — and it doesn't have a one-line Spark function to solve it.

**Problem:**

- Read the JSON with an explicit schema (`manager_id` should allow nulls — that's how you identify the root of the hierarchy, someone with no manager).
- For each employee, compute:
  - `hierarchy_level` — how many steps up the chain they are from the root (the root itself is level `0`).
  - `top_level_manager` — the `name` of the person at the very top of the chain (the root's own `top_level_manager` is their own name).
- Since you don't know the hierarchy's depth in advance, solve this with a **Python `while` loop that repeatedly self-joins** the data one level at a time, rather than hardcoding a fixed number of joins — the loop should keep going until no employee's chain can be extended any further (i.e., every remaining `manager_id` in play resolves to `null`).
- Order the final output by `emp_id` ascending.

**Schema**

| Column | Type |
| :--- | :--- |
| **emp_id** | string |
| **name** | string |
| **manager_id** | string |

**Expected Output**

| emp_id | name | hierarchy_level | top_level_manager |
| :--- | :--- | :--- | :--- |
| E001 | Alice | 0 | Alice |
| E002 | Bob | 1 | Alice |
| E003 | Carol | 1 | Alice |
| E004 | David | 2 | Alice |
| E005 | Eve | 3 | Alice |

In [0]:
schema = StructType(
    [
        StructField("emp_id", StringType(), nullable=False),
        StructField("name", StringType(), nullable=False),
        StructField("manager_id", StringType(), nullable=True),
    ]
)

emp_df = spark.read.schema(schema).json(
    "/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/employees_hierarchy.json"
)
result_df = emp_df.filter(F.col("manager_id").isNull()).select(
    "emp_id",
    "name",
    F.lit(0).alias("hierarchy_level"),
    F.col("name").alias("top_level_manager"),
)
remaining_df = emp_df.filter(F.col("manager_id").isNotNull())

while remaining_df.limit(1).count() > 0:

    manager_df = result_df.select(
        F.col("emp_id").alias("manager_emp_id"),
        F.col("hierarchy_level").alias("manager_level"),
        F.col("top_level_manager").alias("manager_top"),
    )

    current_level_df = (
        remaining_df.alias("e")
        .join(
            manager_df.alias("m"),
            F.col("e.manager_id") == F.col("m.manager_emp_id"),
            "inner",
        )
        .select(
            F.col("e.emp_id"),
            F.col("e.name"),
            (F.col("m.manager_level") + 1).alias("hierarchy_level"),
            F.col("m.manager_top").alias("top_level_manager"),
        )
    )

    result_df = result_df.unionByName(current_level_df)

    remaining_df = remaining_df.join(
        current_level_df.select("emp_id"), "emp_id", "left_anti"
    )

final_df = result_df.orderBy("emp_id")

final_df.show()